# Controlled synthetic factor markets — Gaussian, regimes, and common jumps

This demonstrates the implemented 4A/4B **data-generating process**, independently of forecasting models.
Each source has 12 assets and 8,000 weekday observations: 5,000 training, 1,500 validation, 1,500 held out.
With the existing causal 60-day history / one-day target, usable windows are **4,940 / 1,500 / 1,500**;
validation/test histories may use preceding observations, as in our French experiments.

Log-return simulation follows $x_t=\mu_{z_t}+\sigma_{z_t}(B_{z_t}f_t+D_{z_t}\epsilon_t)-J_t a b_{z_t}$;
$f_t$ and $\epsilon_t$ are independent standard Gaussians. Pipeline returns are arithmetic decimals
$r_t=\operatorname{expm1}(x_t)$, with **no risk-free subtraction**. Prices compound continuously across regimes.
The two-state mixture is not unconditionally Gaussian. Jump probabilities are daily Bernoulli probabilities,
not continuous-time rates; $a=0.04$ is a fixed positive **log-return** event size before asset exposure.

Parameters are illustrative assumptions, not fitted estimates or a new stochastic-model contribution.
Matching configured moments validates implementation; it does not establish empirical market realism.
All diagnostics below use **training dates only**. Hidden regimes/jumps are diagnostic ground truth;
models receive only standardized past returns. No held-out performance evaluation is included.

## Setup

Enable Kaggle Internet. A CPU is sufficient for generation, diagnostics and smoke tests; GPU is optional
for the separate Vanilla pilot at the end. Run ordinary cells interactively. No expensive CDE/Lévy training.
The standard synchronization cell selects the working branch; change `BRANCH` to `main` after merging.

In [ ]:
import os
import sys

REPO_PATH = "/kaggle/working/diffusion-portfolio"
USERNAME = "Amir-Entezari"
REPO = "diffusion-portfolio"
BRANCH = "feature/synthetic-market"

repo_url = f"https://github.com/{USERNAME}/{REPO}.git"

# Clone the specified branch if the repository doesn't exist.
if not os.path.exists(os.path.join(REPO_PATH, ".git")):
    !git clone --branch {BRANCH} --single-branch {repo_url} {REPO_PATH}

else:
    # Fetch latest changes from the specified branch.
    !git -C {REPO_PATH} fetch origin {BRANCH}

    # Switch to the branch and force-sync with remote.
    !git -C {REPO_PATH} checkout -B {BRANCH} FETCH_HEAD
    !git -C {REPO_PATH} reset --hard FETCH_HEAD

os.chdir(REPO_PATH)

SRC_PATH = os.path.join(REPO_PATH, "src")

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Verify the active branch and latest commit.
!git branch --show-current
!git log -1 --oneline

print(f"Repository synced with origin/{BRANCH}.")

In [ ]:
!pip install -q -e '.[dev]'
assert _exit_code == 0, "Dependency installation failed."

## Lightweight mathematical and integration tests

In [ ]:
!pytest -q tests/unit/test_synthetic_market.py tests/unit/test_synthetic_market_pipeline.py \
    tests/unit/test_generic_returns.py tests/semantic/test_data_leakage.py tests/semantic/test_window_continuity.py
assert _exit_code == 0, "Synthetic/data regression tests failed."

## Generate the three specified markets

Source seed 42 is separate from the model-training seed. Defaults use normal factor exposure 0.004–0.008,
idiosyncratic volatility 0.01 and log drift 0.0002 per day. Stress doubles factor loadings, uses idiosyncratic
volatility 0.0075, multiplier 1.5 and log drift -0.0004. Thus factor dependence changes separately from scale.
The transition matrix is `[[0.99, 0.01], [0.04, 0.96]]`: expected complete regime durations 100 and 25 weekdays,
stationary occupancy 80%/20%. Jumps use daily probabilities 0.002/0.03 and default exposure one.

The simulator returns resolved parameters, hidden states, and optional post-return prices separately.
No zero-return initialization row is inserted. Exact regeneration requires the same code and NumPy version;
save the parameter/version record below. The older experimental LOB generator is not used.

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from diffusion_portfolio.config import load_config
from diffusion_portfolio.data.sources.synthetic_market import simulate_market
from diffusion_portfolio.evaluation.synthetic_market import market_diagnostics, plot_market_diagnostics

SCENARIOS = ("synthetic_gaussian", "synthetic_regime", "synthetic_regime_jump")
configs = {name: load_config(f"configs/{name}.yaml") for name in SCENARIOS}
markets = {name: simulate_market(name, **cfg.data.source_options) for name, cfg in configs.items()}
OUT = Path("/kaggle/working/synthetic_market_validation")
OUT.mkdir(exist_ok=True)
for name, result in markets.items():
    print(name, result.returns.shape, result.dates[0].date(), "to", result.dates[-1].date())
    directory = OUT / name
    directory.mkdir(exist_ok=True)
    (directory / "parameters.json").write_text(json.dumps(result.parameters, indent=2, allow_nan=False))
    shutil.copy2(f"configs/{name}.yaml", directory / "config.yaml")

## Configured versus observed training-period statistics

These comparisons use **log returns**, matching the analytical model. Tables report average asset mean,
average asset volatility and average off-diagonal correlation; full per-asset moments/matrices are saved.
Finite-sample estimates need not equal configured values. Regime durations omit both boundary-censored spells.
Jump terms are negative contributions; a simultaneous positive shock can still make the net return positive.

In [ ]:
reports = {}
for name, cfg in configs.items():
    report = market_diagnostics(markets[name], end_date=cfg.data.train_end)
    reports[name] = report
    print(name, report["start"], "to", report["end"], "—", report["units"])
    display(pd.DataFrame(report["summary"]).set_index("regime"))
    (OUT / name / "training_diagnostics.json").write_text(json.dumps(report, indent=2, allow_nan=False))

In [ ]:
moments = reports["synthetic_gaussian"]["moments"]["normal"]
labels = markets["synthetic_gaussian"].columns
display(pd.DataFrame({
    "Configured daily log mean": moments["configured_mean"],
    "Observed daily log mean": moments["observed_mean"],
    "Configured daily log volatility": np.sqrt(np.diag(moments["configured_covariance"])),
    "Observed daily log volatility": np.sqrt(np.diag(moments["observed_covariance"])),
}, index=labels))
print("Configured and observed log-return covariance:")
display(pd.DataFrame(moments["configured_covariance"], index=labels, columns=labels))
display(pd.DataFrame(moments["observed_covariance"], index=labels, columns=labels))

In [ ]:
name = "synthetic_regime_jump"
fig = plot_market_diagnostics(markets[name], end_date=configs[name].data.train_end)
plt.show()
fig.savefig(OUT / name / "training_diagnostics.png", dpi=120, bbox_inches="tight")
report = reports[name]
print("Configured versus observed transition probabilities:")
display(pd.DataFrame(report["configured_transition"], index=["normal", "stress"], columns=["normal", "stress"]))
display(pd.DataFrame(report["observed_transition"], index=["normal", "stress"], columns=["normal", "stress"]))

## The same ReturnTable, normalization and causal windows as real datasets

The adapter returns **only dates, arithmetic returns and asset labels**. Train-only normalization is fitted
on the first 5,000 observations, then saved/restored. Counts below include held-out availability, not test-set
statistics or performance. Ground-truth labels are never concatenated with model inputs.

In [ ]:
from diffusion_portfolio.data.registry import load_return_table
from diffusion_portfolio.data.preparation import prepare_return_data, save_standardizer

rows = []
for name, cfg in configs.items():
    table = load_return_table(name, **cfg.data.source_options)
    assert set(vars(table)) == {"dates", "returns", "columns"}
    np.testing.assert_array_equal(table.returns, markets[name].returns)
    datasets, scaler = prepare_return_data(cfg)
    train_only = table.returns[table.dates <= pd.Timestamp(cfg.data.train_end)]
    np.testing.assert_array_equal(scaler.mean, train_only.mean(0))
    save_standardizer(OUT / name / "standardizer.npz", scaler)
    restored, _ = prepare_return_data(cfg, standardizer_path=OUT / name / "standardizer.npz")
    np.testing.assert_array_equal(restored.val.model_windows.history, datasets.val.model_windows.history)
    rows.append({"Dataset": name, "Assets": len(table.columns), "Train windows": len(datasets.train),
                 "Validation windows": len(datasets.val), "Held-out windows": len(datasets.test),
                 "History shape": tuple(datasets.train[0]["history"].shape),
                 "Target shape": tuple(datasets.train[0]["target"].shape)})
    del datasets, restored
display(pd.DataFrame(rows).set_index("Dataset"))

## Optional manual dataset export

After generation, select this cell to save `observations.npz` (dates, asset labels, arithmetic returns,
and prices when available) and a separate `ground_truth.npz` (aligned dates and hidden diagnostic arrays).
The checked-out full commit SHA is added to `parameters.json`. Files preserve the complete simulated
trajectory, including held-out observations; this is an export, not a test-period evaluation.
Forecasting continues through the existing returns-only source adapter, never the ground-truth archive.

In [ ]:
revision = !git rev-parse HEAD
assert len(revision) == 1, "Could not read the checked-out source revision."
source_revision = revision[0].strip()
assert len(source_revision) == 40 and all(c in "0123456789abcdef" for c in source_revision), "Invalid source revision; export stopped."

for name, result in markets.items():
    directory = OUT / name
    observations = {"dates": result.dates.values, "columns": np.asarray(result.columns), "returns": result.returns}
    if result.prices is not None:
        observations["prices"] = result.prices
    np.savez_compressed(directory / "observations.npz", **observations)
    np.savez_compressed(directory / "ground_truth.npz", dates=result.dates.values, **result.ground_truth)
    parameters = {**result.parameters, "source_revision": source_revision}
    (directory / "parameters.json").write_text(json.dumps(parameters, indent=2, allow_nan=False))
    print("Exported observations and separate ground truth:", directory)

## CPU forward/loss/backward smoke checks

Existing tooling uses **two training examples**, without optimizer updates, sampling or checkpoint writes.
It checks finite gradients and shapes. These losses are smoke diagnostics, not forecasting results.

In [ ]:
!PYTHONUNBUFFERED=1 OMP_NUM_THREADS=1 python -u scripts/data_smoke_test.py \
    --config configs/synthetic_gaussian.yaml --model-smoke --smoke-batch-size 2 --device cpu
assert _exit_code == 0, "synthetic_gaussian model smoke failed."

In [ ]:
!PYTHONUNBUFFERED=1 OMP_NUM_THREADS=1 python -u scripts/data_smoke_test.py \
    --config configs/synthetic_regime.yaml --model-smoke --smoke-batch-size 2 --device cpu
assert _exit_code == 0, "synthetic_regime model smoke failed."

In [ ]:
!PYTHONUNBUFFERED=1 OMP_NUM_THREADS=1 python -u scripts/data_smoke_test.py \
    --config configs/synthetic_regime_jump.yaml --model-smoke --smoke-batch-size 2 --device cpu
assert _exit_code == 0, "synthetic_regime_jump model smoke failed."

## Optional manual Vanilla pilot — two epochs

**Stop here for a generation/diagnostic demonstration.** The next cell trains only when selected manually.
It uses the unchanged Vanilla optimizer/objective and absolute epoch boundary 2; rerunning resumes the same
isolated synthetic run. No Neural CDE, Lévy or held-out evaluation. The configured full schedule is 50 epochs,
but this command stops at 2. Keep pilot conclusions limited to numerical/computational feasibility.

In [ ]:
from diffusion_portfolio.utils.research_checks import check_checkpoint_pair
RUN_DIR = Path("/kaggle/working/synthetic_gaussian_vanilla")
if (RUN_DIR / "config.yaml").exists():
    assert load_config(RUN_DIR / "config.yaml") == configs["synthetic_gaussian"], "Existing run has another configuration."
check_checkpoint_pair(RUN_DIR)

!PYTHONUNBUFFERED=1 OMP_NUM_THREADS=1 python -u scripts/train_diffusion.py \
    --config configs/synthetic_gaussian.yaml \
    --output-dir /kaggle/working/synthetic_gaussian_vanilla \
    --device cpu --resume --stop-after-epoch 2
assert _exit_code == 0, "Optional Vanilla pilot failed."

In [ ]:
import torch
if (RUN_DIR / "last.pt").exists():
    history = pd.DataFrame(torch.load(RUN_DIR / "last.pt", map_location="cpu", weights_only=False)["history"])
    ax = history.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", title="Synthetic Gaussian — Vanilla pilot")
    ax.set(xlabel="Epoch", ylabel="Diffusion loss")
    ax.grid(True, alpha=.3)
    ax.legend(["Train Loss", "Validation Loss"])
    plt.show()
    display(history)

## Interpretation and saved records

`/kaggle/working/synthetic_market_validation/<scenario>/` contains the exact configuration, resolved
parameters/NumPy version, saved scaler and **training-only** diagnostic report (plus the jump-market figure).
Copy these outputs using Kaggle's normal Save Version/download workflow; this notebook does not upload or
replace the central financial artifact dataset. Optional pilot checkpoints have their own directory.

No forecasting superiority or empirical financial realism is asserted. The return process is conditionally
Gaussian with fixed state parameters and optional discrete common jumps; it has no stochastic-volatility
recursion or exchange holiday calendar. A mixture of regimes/jumps is not unconditionally Gaussian.

Future 4C scenarios and separate topology fixtures, 4D irregular/intraday/event data, and broader downstream
forecasting/portfolio studies remain unimplemented. Select their scientific protocols separately.